In [ ]:
import sys, os, glob, shutil, time, json, gc
import numpy as np, pandas as pd
t0 = time.perf_counter()
def log(m): print(f"[{time.perf_counter()-t0:6.0f}с] {m}", flush=True)
code = os.path.dirname(glob.glob("/kaggle/input/**/pair_features.py", recursive=True)[0])
os.makedirs("/kaggle/working/src", exist_ok=True)
for p in glob.glob(code + "/*.py"): shutil.copy(p, "/kaggle/working/src/")
open("/kaggle/working/src/__init__.py", "a").close()
os.makedirs("/kaggle/working/models", exist_ok=True)
for p in glob.glob(code + "/*.json"): shutil.copy(p, "/kaggle/working/models/")
os.chdir("/kaggle/working"); sys.path.insert(0, "/kaggle/working")
from src.pair_features import build_matrix, feature_names
from src.measure_features import measures, compare_measures, MEASURE_FEATURES
from src.export_boost import export, save, predict_proba
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import average_precision_score
from scipy.stats import rankdata

fold = os.path.dirname(glob.glob("/kaggle/input/**/llm_valid_pairs.parquet", recursive=True)[0])
sc = os.path.dirname(glob.glob("/kaggle/input/**/ce_relaxed.npy", recursive=True)[0])
pairs = pd.read_parquet(fold + "/llm_valid_pairs.parquet")
items = pd.read_parquet(fold + "/llm_valid_items.parquet")
y = (pairs["target"].to_numpy() > 0).astype(np.int8)
cat = pairs["id1"].map(dict(zip(items.id, items.category.astype(str)))).astype(str).to_numpy()
known = sorted(set(items.category.astype(str)))
log(f"пар {len(pairs):,}, доля+ {y.mean():.3f}")

names = list(feature_names(False))
X = np.zeros((len(pairs), len(names)), dtype=np.float32)
for c in known:
    rows = np.flatnonzero(cat == c)
    if not len(rows): continue
    sub = items[items.category.astype(str) == c].reset_index(drop=True)
    X[rows] = build_matrix(sub, pairs.iloc[rows].reset_index(drop=True), known, with_neighbours=False)
    del sub; gc.collect()
log("признаки готовы")
M = {int(i): measures(a) for i, a in zip(items.id, items.attributes)}
MX = np.array([[r[n] for n in MEASURE_FEATURES] for r in
               (compare_measures(M[x], M[z]) for x, z in zip(pairs.id1, pairs.id2))], dtype=np.float32)
CE = {n: np.load(f"{sc}/{n}.npy").astype(np.float32) for n in ("ce_relaxed", "ce_combo")}
code_cat = np.array([known.index(c) if c in known else -1 for c in cat], dtype=np.float32)

# Порядок столбцов боевой модели. Инференс обязан собрать ровно этот же.
COLUMNS = names + list(MEASURE_FEATURES) + ["ce_relaxed", "ce_combo", "category_code"]
FULL = np.column_stack([X, MX, CE["ce_relaxed"], CE["ce_combo"], code_cat]).astype(np.float64)
log(f"матрица {FULL.shape}, столбцов {len(COLUMNS)}")

masks = {c: cat == c for c in np.unique(cat)}
def rk(s):
    o = np.empty(len(s), np.float32)
    for m in masks.values(): o[m] = rankdata(s[m]) / m.sum()
    return o
def macro(s, rate=0.111, seeds=8):
    vals = []
    for seed in range(seeds):
        rng = np.random.default_rng(seed); per = []
        for m in masks.values():
            rows = np.flatnonzero(m); pos, neg = rows[y[rows]==1], rows[y[rows]==0]
            keep = min(len(pos), max(5, int(round(rate/(1-rate)*len(neg)))))
            ch = np.concatenate([rng.choice(pos, keep, replace=False), neg])
            per.append(average_precision_score(y[ch], s[ch]))
        vals.append(np.mean(per))
    return float(np.mean(vals)), float(np.std(vals))

enc = 0.56*rk(CE["ce_relaxed"]) + 0.44*rk(CE["ce_combo"])
log(f"только энкодеры: {macro(enc)[0]:.6f}")

PARAMS = dict(max_iter=500, max_leaf_nodes=63, learning_rate=0.06, l2_regularization=1.0,
              early_stopping=False, random_state=0)
half = np.random.default_rng(5).permutation(len(y)) % 2
oof = np.zeros(len(y))
for h in (0, 1):
    tr, te = half != h, half == h
    g = HistGradientBoostingClassifier(**PARAMS).fit(FULL[tr], y[tr])
    oof[te] = g.predict_proba(FULL[te])[:, 1]
mu, sd = macro(rk(oof))
log(f"слитая модель, честная проверка: {mu:.6f} ± {sd:.6f}")

final = HistGradientBoostingClassifier(**PARAMS).fit(FULL, y)
trees = export(final)
save("/kaggle/working/fusion_boost.npz", trees)
check = predict_proba(trees, FULL[:2000])
same = np.allclose(check, final.predict_proba(FULL[:2000])[:, 1], atol=1e-6)
log(f"выгрузка совпадает с моделью: {same}")
json.dump({"columns": COLUMNS, "categories": known, "params": PARAMS,
           "honest_macro": mu, "n_train": int(len(y))},
          open("/kaggle/working/fusion_info.json", "w"), ensure_ascii=False, indent=1)
log("готово")
